# GRCh38 Verification

This notebook is used as a verification tool to make sure MUTACC hg19 solved cases behave identically on
GRCh38 in raredisease v3.0.

## Input required
- Subset of MUTACC solved cases from CG, see this [samplesheet](https://github.com/Clinical-Genomics/mivmirvalidation/blob/master/samplesheet.csv)
- Above grch37 cases run through the mivmir validation pipeline, to annotate variants with mivmir gicam scores (grch37 inferences)
- Subset of above cases where access to fastq data is available, to run raredisease v3 with mivmir gicam annotation enabled (to generate grch38 inferences)

```
/rdds/grch38-verification/
    - grch38
        - CASE
        - ...
    - hg19
        - CASE.vcf
        - causatives/
            - CASE.vcf
    - references/
```

## 1. Convert old MIP cases to hg38

In [164]:
%load_ext autoreload
%autoreload all -l

In [134]:
DATA_DIR='/rdds/grch38-verification'

In [8]:
%%script bash
cat /etc/os*

NAME="Ubuntu"
VERSION="20.04.6 LTS (Focal Fossa)"
ID=ubuntu
ID_LIKE=debian
PRETTY_NAME="Ubuntu 20.04.6 LTS"
VERSION_ID="20.04"
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
VERSION_CODENAME=focal
UBUNTU_CODENAME=focal


In [112]:
from rdds.lib.git import git_version; git_version()

'v1.12.1-35-g6b9b554-dirty'

In [ ]:
%%script bash
# https://broadinstitute.github.io/picard/
apt-get update && apt-get install -y picard-tools

In [ ]:
# For MUTACC causative variants, remove INFO/END as it's conflicting with coordinates in the new genome ref
# for csv in `realpath *`; do sed -i 's/END\=[0123456789]*\;//g'  $csv &&  sed -i 's/\#\#INFO=<ID=END.*//g' $csv ; done
# FIXME: No empty line in VCF header - picard tools complains; sed -r '/^\s*$/d'

In [ ]:
%%script bash
# Convert hg19 MUTACC solved causatives to grch38

#set -e
# https://github.com/broadinstitute/picard/issues/1258
# Manual conversion: https://liftover.broadinstitute.org/#input=chr19%3A38937347%20G%3ET&hg=hg19-to-hg38

for case_vcf in `realpath /rdds/grch38-verification/hg19/causatives/selected-subset/*.vcf`; do
    echo Processing $case_vcf
    VCF_NAME=`basename $case_vcf`
    OUT=`dirname $case_vcf`/as-grch38/$VCF_NAME
    OUT_REJECTED=$OUT_DIR/rejected/$VCF_NAME

    mkdir -p `dirname $OUT`
    mkdir -p `dirname $OUT_REJECTED`

    picard-tools LiftoverVcf \
    I=$case_vcf \
    O=$OUT \
    REJECT=$OUT_REJECTED \
    CHAIN=/rdds/grch38-verification/references/hg19ToHg38_nochr.over.chain \
    REFERENCE_SEQUENCE=/rdds/grch38-verification/references/hg38.p14.fa.gz \
    VERBOSITY=ERROR \
    WARN_ON_MISSING_CONTIG=true \
    LIFTOVER_MIN_MATCH=0.95
done


In [ ]:
%%script bash
# Convert hg19 inferences to grch38
set -e

for case_vcf in `realpath /rdds/grch38-verification/hg19/*.vcf`; do
    echo Processing $case_vcf
    VCF_NAME=`basename $case_vcf`
    OUT=`dirname $case_vcf`/as-grch38/$VCF_NAME
    OUT_REJECTED=$OUT_DIR/rejected/$VCF_NAME

    mkdir -p `dirname $OUT`
    mkdir -p `dirname $OUT_REJECTED`

    picard-tools LiftoverVcf \
    I=$case_vcf \
    O=$OUT \
    REJECT=$OUT_REJECTED \
    CHAIN=/rdds/grch38-verification/references/hg19ToHg38_nochr.over.chain \
    REFERENCE_SEQUENCE=/rdds/grch38-verification/references/hg38.p14.fa.gz \
    VERBOSITY=DEBUG \
    WARN_ON_MISSING_CONTIG=true \
    LIFTOVER_MIN_MATCH=0.95
done


# 2. Compute Performance Metrics

Goal: to check causative variant rank in raredisease v3 grch38 equal to that of mivmirvalidation pipeline.

## How to compare MUTACC solved case VCFs to output from raredisease v3

The MUTACC solved cases are filtered before upload, to remove common variants. This is not the case in the VCFs coming from RD pipeline.
In order to make them comparable, drop the additional N variants in RD VCF not in MUTACC VCF.
Furthermore, there might be a difference in the gene panel applied in mivmirvalidation and RDv3,
as the mivmirvalidation pipeline is a custom pipeline not part of RDv3 and CG tooling.

## How to uniquely identify the causative variant

In the hg19 VCFs, converted to grch38, the ID is preserved as the hg19.
In the RD VCFs, the ID is based on the grch38 position. This makes the ID incomparable.
-> There's a need for a custom ID.

Furthermore, in the causative VCFs converted to grch38, there's no ID, it's set to missing `.`.

In [2]:
# Selected subset of cases for verification
hg19_grch38_map = {
'actualgriffon': 'engagedmite',
'allowedsawfly':'lenientchimp',
'evidentpony':'lastinggrizzly',
'factualnewt':'ablecrow',
'finerram': 'mainantelope',
'genuinereptile': 'relaxedkiwi',
'informedmollusk': 'preciousmuskox',
'nexthare':'humblecow',
#'patientllama':'learningrooster', Could not retrieve fastq data from CG backend
'precisespaniel':'amplegrouper'
}

In [ ]:
from glob import glob
import os
import pandas as pd
from tempfile import mkdtemp
from progressbar import ProgressBar
import matplotlib.pyplot as plt
import progressbar.widgets
import numpy as np
from cyvcf2 import Writer
import gc

from rdds.variant_rank_score.inference_exploration.view_vcf_rank_results import view_vcf_rank_results
from rdds.variant_rank_score.inference_exploration.statfns import plot_performance_vs_threshold
from rdds.lib.vcf import VCFReader, ParsableVariant

In [ ]:
def vcf_to_pandas(vcf_path: str, is_causative=False) -> pd.DataFrame:
    vcf_reader = VCFReader(vcf_path)
    csq_desc = vcf_reader.csq_description
    n_variants = vcf_reader.number_of_variants
    print(f"{vcf_path}: {n_variants} variants")
    chrom = []
    pos = []
    alt = []
    ref = []
    mivmir = []
    gicam = []
    genmod = []
    parse_only_fields = ['POS', 'CHROM', 'RankScoreNormalized', 'MivmirScore', 'GicamScore']
    pbar = ProgressBar(max_value=n_variants)
    for i, variant in enumerate(vcf_reader):
        parsed_variant = ParsableVariant(variant=variant,
                                         parse_only_fields=parse_only_fields,
                                         vep_csq_description=vcf_reader.csq_description)
        chrom.append(parsed_variant.CHROM)
        pos.append(int(parsed_variant.POS))
        ref.append(parsed_variant.REF)
        alt.append(parsed_variant.ALT)
        try:
            mivmir.append(parsed_variant.MivmirScore)
            gicam.append(parsed_variant.GicamScore)
            genmod.append(parsed_variant.RankScoreNormalized_value)
        except AttributeError:
            pass
        pbar.update(i)
    pbar.finish()

    df = pd.DataFrame({
        'chrom': chrom,
        'pos': pos,
        'ref': ref,
        'alt': alt,
        'mivmir': mivmir if len(mivmir) == n_variants else None,
        'gicam': gicam if len(gicam) == n_variants else None, 
        'genmod': genmod if len(genmod) == n_variants else None,
        'causative': [1.0] * n_variants if is_causative else [0.0] * n_variants
    })

    # Store variant vcf position
    df['vcf_index'] = df.index
    
    # Set index to custom ID
    df.index = df.apply(lambda row: f"{row.chrom}_{row.pos}-{row.ref}-{row.alt}", axis=1)
    
    return df
        

In [350]:
def _write_variants_to_vcf(indexes: np.ndarray,
                           file_name: str,
                           output_file_name: str):
    """
    Copy variants from input VCF to a new VCF as a subset based on indexes
    """
    pbar = ProgressBar(widgets=[progressbar.widgets.BouncingBar()],
                       prefix=file_name)
    pbar.start()
    indexes = list(indexes)
    vcf_reader = VCFReader(fname=file_name)
    vcf_writer: Writer = Writer(output_file_name, vcf_reader, mode='w')
    variants = list(vcf_reader)  # Load to RAM
    # Write variants in sorted order to output VCF
    for index in indexes:
        vcf_writer.write_record(variants[index])
    vcf_reader.close()
    vcf_writer.close()
    del variants
    gc.collect()
    pbar.finish()

In [368]:
def compute_metrics_per_case(mivmirval_case_name, raredisease_case_name):
    print(f"hg19 {mivmirval_case_name}:{raredisease_case_name} grch38")
    mixed_case_name = f"{mivmirval_case_name}-{raredisease_case_name}"
    base_work_dir = f'/rdds/tmp/grch38-verification/{mixed_case_name}'
    os.makedirs(base_work_dir, exist_ok=True)
    ref_dir = base_work_dir + '/mivmirvalidation'
    rd_dir = base_work_dir + '/raredisease_v3'
    print(base_work_dir)
    causative_vcf_as_grch38 = os.path.join(DATA_DIR, 'hg19', 'causatives', 'selected-subset', 'as-grch38', f'{mivmirval_case_name}_causative.vcf')
    assert os.path.exists(causative_vcf_as_grch38), causative_vcf_as_grch38
    vcf_mivmir_validation = glob(os.path.join(DATA_DIR, 'hg19', 'as-grch38', mivmirval_case_name + '*.vcf'))[0]
    assert os.path.exists(vcf_mivmir_validation), vcf_mivmir_validation
    vcf_raredisease_grch38 = os.path.join(DATA_DIR, 'grch38', raredisease_case_name, 'rank_and_filter', f'{raredisease_case_name}_snv_ranked_clinical.vcf.gz')
    assert os.path.exists(vcf_raredisease_grch38), vcf_raredisease_grch38
    #view_vcf_rank_results(vcf_file_path=vcf_mivmir_validation, vcf_pathogenic_path=causative_vcf_as_grch38, workdir=ref_dir)
    #view_vcf_rank_results(vcf_file_path=vcf_raredisease_grch38, vcf_pathogenic_path=causative_vcf_as_grch38, workdir=rd_dir)
    df_causative = vcf_to_pandas(causative_vcf_as_grch38, is_causative=True)
    assert len(df_causative) == 1, len(df_causative)
    df_hg19 = vcf_to_pandas(vcf_mivmir_validation)
    df_grch38 = vcf_to_pandas(vcf_raredisease_grch38)

    df_causative.to_csv(os.path.join(base_work_dir, 'causative.csv'))

    # Store sorted grch38 variants VCF
    store_n_variants = 4000
    for model in ['gicam', 'genmod']:
        _write_variants_to_vcf(indexes=df_grch38.set_index('vcf_index').sort_values(model,
                               ascending=False).iloc[0:store_n_variants].index.values,
                               file_name=vcf_raredisease_grch38,
                               output_file_name=os.path.join(base_work_dir, os.path.basename(vcf_raredisease_grch38).replace('.vcf', f'-grch38-sorted-{model}.vcf')))

    # Set causative variant label
    df_hg19.loc[df_causative.index[0], 'causative'] = 1.0
    df_grch38.loc[df_causative.index[0], 'causative'] = 1.0
    assert len(df_hg19[df_hg19.causative == 1]) == 1
    assert len(df_grch38[df_grch38.causative == 1]) == 1

    # Performance GRCH38 per model on all of variants in grch38 VCF
    for model in ['gicam', 'mivmir', 'genmod']:
        plot_performance_vs_threshold(predictions=df_grch38[f'{model}'].values,
                                     labels=df_grch38.causative.values,
                                     output_path=os.path.join(base_work_dir, f'performance-{model}-all-grch38-variants.png'))
    
    # Merge VCFs based on the intersection of index CHR_POS_REF_ALT in the VCFs (dropping non common variants)
    # This is NOT merging index on different reference genome positions, the intersection refers to variants overlapping MUTACC excerpt and rd v3 fastq inferences
    cols = ['genmod', 'gicam', 'mivmir', 'causative']
    merged_hg19_grch38 = df_hg19[cols].merge(df_grch38[cols], how='inner', left_index=True, right_index=True, suffixes=('_hg19', '_grch38'))

    # Compute performance on intersecting variants
    for ref_genome in ['hg19', 'grch38']:
        for model in ['genmod', 'mivmir', 'gicam']:
            plot_performance_vs_threshold(predictions=merged_hg19_grch38[f'{model}_{ref_genome}'].values,
                                         labels=merged_hg19_grch38[f'causative_{ref_genome}'].values,
                                         output_path=os.path.join(base_work_dir, f'performance-intersecting-variants-{model}-{ref_genome}.png'))

In [ ]:
import traceback
import sys

for mivmirval_case_name, raredisease_case_name in hg19_grch38_map.items():
    try:
        compute_metrics_per_case(mivmirval_case_name, raredisease_case_name)
    except Exception as e:
        print('ERROR', mivmirval_case_name, raredisease_case_name , e)
        traceback.print_exc(file=sys.stdout)


In [ ]:
from glob import glob
from IPython.display import Image
genmod_perf_plots = glob('/rdds/tmp/grch38-verification/**/*genmod-all-grch38-variants.png')
gicam_perf_plots = glob('/rdds/tmp/grch38-verification/**/*gicam-all-grch38-variants.png')
for genmod_plot, gicam_plot in zip(genmod_perf_plots, gicam_perf_plots):
    print(genmod_plot)
    display(Image(genmod_plot))
    print(gicam_plot)
    display(Image(gicam_plot))

In [ ]:
from rdds.variant_rank_score.model.model import FEATURES_TEXT, FEATURES_FLOAT
MODEL_FEATURES = []
MODEL_FEATURES.extend(FEATURES_TEXT)
MODEL_FEATURES.extend(FEATURES_FLOAT)
MODEL_FEATURES.append('GNOMADAF_grpmax')  # Replaces GNOMADAF_popmax in grch38
MODEL_FEATURES.append('MivmirScore')  # Store inference to CSV as well
MODEL_FEATURES.append('GicamScore')

In [ ]:
# Load causative variants and compare feature inputs

def variant_id(variant) -> str:
    return f"{variant.CHROM}-{variant.POS}-{variant.REF}-{variant.ALT}"

for mivmirval_case_name, raredisease_case_name in hg19_grch38_map.items():
    mixed_case_name = f"{mivmirval_case_name}-{raredisease_case_name}"
    base_work_dir = f'/rdds/tmp/grch38-verification/{mixed_case_name}'
    print(mixed_case_name)
    df_path = os.path.join(base_work_dir, 'comparison-input-features.csv')
    if os.path.exists(df_path):
        print('Comparison exists, continuing.')
        continue
    causative_file_hg19 = glob(f'/rdds/grch38-verification/hg19/causatives/selected-subset/as-grch38/{mivmirval_case_name}_causative.vcf')[0]
    causative_reader= VCFReader(causative_file_hg19, unpack_if_gzipped=False)
    assert causative_reader.number_of_variants == 1
    causative_variant = list(causative_reader)[0]
    causative_reader.close()
    causative_variant_id = variant_id(causative_variant)
    print('causative:', causative_file_hg19, causative_variant_id)
    # Load mivmirval and raredisease v3 inferences
    print('Reading mivmirval data ...')
    hg19_case_vcf = glob(f'/rdds/grch38-verification/hg19/as-grch38/{mivmirval_case_name}*.vcf')[0]
    print(hg19_case_vcf)
    mivmirval_reader = VCFReader(hg19_case_vcf, unpack_if_gzipped=False)
    mivmirval_causative = None
    for variant in mivmirval_reader:
        if variant_id(variant) == causative_variant_id:
            print('match mivmirval', variant.CHROM, causative_variant_id, variant_id(variant))
            mivmirval_causative = ParsableVariant(variant, vep_csq_description=mivmirval_reader.csq_description)
            break
    assert mivmirval_causative is not None
    mivmirval_reader.close()
    print('Reading raredisease v3 data ...')
    grch38_case_vcf = glob(f'/rdds/grch38-verification/grch38/{raredisease_case_name}/rank_and_filter/{raredisease_case_name}_snv_ranked_clinical.vcf.gz')[0]
    print(grch38_case_vcf)
    raredisease_reader = VCFReader(grch38_case_vcf, unpack_if_gzipped=False)
    raredisease_causative = None
    for variant in raredisease_reader:
        if variant_id(variant) == causative_variant_id:
            print('match RD', variant.CHROM, causative_variant_id, variant_id(variant))
            raredisease_causative = ParsableVariant(variant, vep_csq_description=raredisease_reader.csq_description)
            break
    assert raredisease_causative is not None
    raredisease_reader.close()

    df = pd.DataFrame()
    for feature_name in MODEL_FEATURES:
        try:
            mivmireval_feature = getattr(mivmirval_causative, feature_name)
        except AttributeError:
            mivmireval_feature = None
        try:
            raredisease_feature = getattr(raredisease_causative, feature_name)
        except AttributeError:
            raredisease_feature = None
            
        _df = pd.DataFrame(data = {
            feature_name: [mivmireval_feature, raredisease_feature]
        },
                          index=['mivmireval-'+variant_id(mivmirval_causative), 'rd-'+variant_id(mivmirval_causative)])
        df = pd.concat((df, _df), axis=1)  # concat columnwise    
    print('storing', df_path)
    df.to_csv(df_path)


In [ ]:
df_comparisons = glob('/rdds/tmp/grch38-verification/*/comparison-input-features.csv', recursive=True)
pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.max_colwidth = None
for df_path in df_comparisons:
    print(df_path)
    display(pd.read_csv(df_path))

In [ ]:
# transfer images, preserving catalog names: rsync -v --relative vm:CATALOG_DIR/tmp/grch38-verification/./**/*.png .